In [ ]:
# model_single_run_v2.py
# ─────────────────────────────────────────────────────────────────────────────
# InLegalBERT + Signal-Cross-Attention + MHA + BiLSTM
# Full 5000 docs · 50 epochs · Resume-safe · Full metrics + adaptive HP
#
# ══ ANTI-OVERFITTING CHANGES (v2) ═══════════════════════════════════════════
#  1.  DROPOUT 0.1 → 0.4  |  LSTM_DROPOUT 0.1 → 0.3  |  MHA_DROPOUT 0.1 → 0.3
#  2.  LSTM_HIDDEN 256 → 128  |  LSTM_LAYERS 2 → 1  |  MHA_HEADS 8 → 4
#  3.  LR_BERT 2e-5 → 5e-6   |  LR_HEAD 1e-5 → 5e-6  |  WEIGHT_DECAY 0.01 → 0.05
#  4.  FREEZE_BERT_LAYERS 6 → 10  (only top-2 BERT layers trained)
#  5.  AdaptiveHP auto-unfreeze DISABLED (was worsening overfit)
#  6.  CHUNK_DROP_PROB 0.15 → 0.35
#  7.  LABEL_SMOOTHING 0.05 → 0.15
#  8.  Head weight_decay → 0.10  (separate from BERT weight_decay)
#  9.  MAX_CHUNKS 4 → 2          (halves param-to-data ratio)
#  10. ACCUM_STEPS 2 → 4  |  BATCH_SIZE 8 → 4  (effective batch=16, smoother)
#  11. SWA_START 35 → 10   |  SWA_LR 5e-6 → 1e-6
#  12. DEFERRED_RW_EPOCH 6 → 1  (class weights active from epoch 1)
#
# ══ ORIGINAL FEATURES (unchanged) ══════════════════════════════════════════
#  · SIGNAL CROSS-ATTENTION
#  · LAYER-WISE LR DECAY (LLRD)
#  · STOCHASTIC WEIGHT AVERAGING (SWA)
#  · FULL EVALUATION METRICS every epoch
#  · RESUME-SAFE CHECKPOINT
#  · All BERT layers pre-registered in optimizer (no runtime add_param_group)
# ─────────────────────────────────────────────────────────────────────────────

import os, gc, json, random, logging, warnings, csv, math
from copy import deepcopy
from datetime import datetime
from collections import defaultdict, Counter
from typing import Optional

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
INPUT_PATH = "Track_A_qa_judgment_flat_OLLAMA.jsonl"
OUTPUT_DIR = "single_run_results_INcaselawbert"
LOG_DIR    = f"{OUTPUT_DIR}/logs"
PLOT_DIR   = f"{OUTPUT_DIR}/plots"
CKPT_ROLL  = f"{OUTPUT_DIR}/checkpoint_last.pt"
CKPT_BEST  = f"{OUTPUT_DIR}/best_model.pt"
CSV_PATH   = f"{OUTPUT_DIR}/epoch_results.csv"

INLEGAL_MODEL_ID = "law-ai/InCaseLawBERT"

MAX_TOTAL_DOCS = 5000
MAX_EPOCHS     = 50
EARLY_STOP_PAT = 15
BATCH_SIZE     = 4          # v2: was 8 → smoother gradient signal
ACCUM_STEPS    = 4          # v2: was 2 → effective batch = 16 (same), but smoother

# ── Learning rates & regularisation ──────────────────────────────────────────
LR_BERT        = 5e-6       # v2: was 2e-5
LR_HEAD        = 5e-6       # v2: was 1e-5
LLRD_DECAY     = 0.95
WARMUP_RATIO   = 0.06
WEIGHT_DECAY   = 0.05       # v2: was 0.01  (BERT + other groups)
HEAD_WEIGHT_DECAY = 0.10    # v2: NEW — extra L2 on head components

# ── Architecture ──────────────────────────────────────────────────────────────
MAX_CHUNK_LEN      = 256
MAX_CHUNKS         = 2      # v2: was 4 — halves param-to-data ratio
LSTM_HIDDEN        = 128    # v2: was 256
LSTM_LAYERS        = 1      # v2: was 2
LSTM_DROPOUT       = 0.3    # v2: was 0.1
MHA_HEADS          = 4      # v2: was 8
MHA_DROPOUT        = 0.3    # v2: was 0.1
DROPOUT            = 0.4    # v2: was 0.1
FREEZE_BERT_LAYERS = 10     # v2: was 6 — only top-2 BERT layers trained

# ── Training tricks ───────────────────────────────────────────────────────────
WITH_SIGNAL         = True
LABEL_SMOOTHING     = 0.15  # v2: was 0.05
DEFERRED_RW_EPOCH   = 1     # v2: was 6 — class weights active from epoch 1
CHUNK_DROP_PROB     = 0.35  # v2: was 0.15
SWA_START           = 10    # v2: was 35
SWA_LR              = 1e-6  # v2: was 5e-6

# ── AdaptiveHP thresholds ────────────────────────────────────────────────────
OVERFIT_GAP_THRESH  = 0.15
OVERFIT_PATIENCE    = 3
# NOTE: UNDERFIT_F1_THRESH kept but auto-unfreeze is DISABLED in v2
UNDERFIT_F1_THRESH  = 0.55

SEED     = 42
SOTA_F1  = 0.8131
SOTA_ACC = 0.78
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP  = DEVICE == "cuda"

for d in [OUTPUT_DIR, LOG_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════
run_id   = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"{LOG_DIR}/run_{run_id}.log"
logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s | %(message)s",
    datefmt  = "%H:%M:%S",
    handlers = [logging.FileHandler(log_file), logging.StreamHandler()],
)
log = logging.getLogger()
log.info(f"Device        : {DEVICE}  |  AMP: {USE_AMP}")
log.info(f"Architecture  : InLegalBERT → SignalCrossAttn → MHA({MHA_HEADS}h) → BiLSTM({LSTM_HIDDEN}h,{LSTM_LAYERS}L) → AttnPool → Linear")
log.info(f"Epochs        : {MAX_EPOCHS}  patience={EARLY_STOP_PAT}  SWA from ep {SWA_START}")
log.info(f"LR BERT/HEAD  : {LR_BERT}/{LR_HEAD}  LLRD={LLRD_DECAY}  WD={WEIGHT_DECAY}  HeadWD={HEAD_WEIGHT_DECAY}")
log.info(f"Dropout       : main={DROPOUT}  lstm={LSTM_DROPOUT}  mha={MHA_DROPOUT}")
log.info(f"MAX_CHUNKS    : {MAX_CHUNKS}  CHUNK_DROP={CHUNK_DROP_PROB}  LABEL_SMOOTH={LABEL_SMOOTHING}")
log.info(f"FREEZE_BERT   : {FREEZE_BERT_LAYERS} layers  (auto-unfreeze DISABLED)")
log.info(f"DEFERRED_RW   : ep {DEFERRED_RW_EPOCH} (effective from start)")
log.info(f"v2 changes    : dropout↑ | complexity↓ | LR↓ | WD↑ | freeze↑ | SWA early | class-weights early")


# ══════════════════════════════════════════════════════════════════════════════
# SEED
# ══════════════════════════════════════════════════════════════════════════════
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(SEED)
torch.backends.cudnn.enabled   = True
torch.backends.cudnn.benchmark = True


# ══════════════════════════════════════════════════════════════════════════════
# SIGNAL TOKENS
# ══════════════════════════════════════════════════════════════════════════════
SIGNAL_MAP = {
    "FAVORS_PETITIONER": "[FP]",
    "FAVORS_RESPONDENT": "[FR]",
    "NEUTRAL"          : "[N]",
}
SIGNAL_IDX = {"FAVORS_PETITIONER": 0, "FAVORS_RESPONDENT": 1, "NEUTRAL": 2}

def format_input(question, answer, signal, with_signal=True):
    sig = SIGNAL_MAP.get(signal, "[N]") if with_signal else ""
    return f"Q: {question.strip()} A: {answer.strip()} {sig}".strip()


# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════
class HierarchicalInLegalBERT(nn.Module):
    """
    InLegalBERT + Signal-Cross-Attention + MHA + BiLSTM + Attn-Pool

    v2 changes:
      - Reduced MHA heads (8 → 4), LSTM hidden (256 → 128), LSTM layers (2 → 1)
      - Increased all dropout values
      - unfreeze_bert_from() still exists but AdaptiveHP will NOT call it
    """

    def __init__(self, model_id, num_labels=2, dropout=0.4,
                 lstm_hidden=128, lstm_layers=1, lstm_dropout=0.3,
                 mha_heads=4, mha_dropout=0.3,
                 label_smoothing=0.15, freeze_bert_layers=10):
        super().__init__()
        self.label_smoothing    = label_smoothing
        self.freeze_bert_layers = freeze_bert_layers

        self.bert = AutoModel.from_pretrained(model_id)
        D = self.bert.config.hidden_size   # 768
        self._freeze_bert(freeze_bert_layers)

        self.signal_emb = nn.Embedding(3, D)
        nn.init.normal_(self.signal_emb.weight, std=0.02)

        # v2: mha_heads=4, mha_dropout=0.3
        self.signal_cross_attn = nn.MultiheadAttention(
            embed_dim=D, num_heads=mha_heads,
            dropout=mha_dropout, batch_first=True,
        )
        self.signal_norm = nn.LayerNorm(D)

        # v2: mha_heads=4, mha_dropout=0.3
        self.chunk_mha   = nn.MultiheadAttention(
            embed_dim=D, num_heads=mha_heads,
            dropout=mha_dropout, batch_first=True,
        )
        self.mha_norm    = nn.LayerNorm(D)
        self.mha_dropout = nn.Dropout(mha_dropout)

        # v2: lstm_hidden=128, lstm_layers=1, lstm_dropout=0.3
        # NOTE: nn.LSTM dropout param is between layers; with layers=1 it is
        #       effectively 0 inside LSTM. We apply our own dropout after.
        bilstm_out = lstm_hidden * 2
        self.bilstm = nn.LSTM(
            input_size=D, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            bidirectional=True,
            dropout=lstm_dropout if lstm_layers > 1 else 0.0,
        )
        # Extra output dropout after BiLSTM (compensates lstm_layers=1 not
        # having inter-layer dropout)
        self.lstm_out_dropout = nn.Dropout(lstm_dropout)

        self.attn_layer = nn.Linear(bilstm_out, 1)
        self.dropout    = nn.Dropout(dropout)   # v2: 0.4
        self.classifier = nn.Linear(bilstm_out, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def _freeze_bert(self, n_layers):
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = (i >= n_layers)

    def unfreeze_bert_from(self, n_layers):
        """
        Kept for API compatibility — but AdaptiveHP v2 does NOT call this.
        All BERT layers are still pre-registered in the optimizer.
        """
        self._freeze_bert(n_layers)
        self.freeze_bert_layers = n_layers
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        log.info(f"  [unfreeze_bert_from] layer≥{n_layers} "
                 f"→ {trainable:,} trainable params")

    def encode_chunks(self, chunk_input_ids, chunk_attention_mask, chunk_mask):
        B, N, L   = chunk_input_ids.shape
        flat_ids  = chunk_input_ids.view(B * N, L)
        flat_mask = chunk_attention_mask.view(B * N, L)
        out = self.bert(input_ids=flat_ids, attention_mask=flat_mask)
        cls = out.last_hidden_state[:, 0, :].view(B, N, -1)
        cls = cls * chunk_mask.unsqueeze(-1).float()
        return cls

    def forward(self, chunk_input_ids, chunk_attention_mask, chunk_mask,
                signal_ids, labels=None, chunk_drop_prob=0.0):

        chunk_cls = self.encode_chunks(
            chunk_input_ids, chunk_attention_mask, chunk_mask)

        # Dynamic chunk dropout (v2: prob=0.35)
        if chunk_drop_prob > 0.0 and self.training:
            drop_mask   = (torch.rand(chunk_cls.shape[:2],
                                      device=chunk_cls.device) > chunk_drop_prob)
            safe_mask   = chunk_mask.bool() & drop_mask
            any_real    = safe_mask.any(dim=1, keepdim=True)
            final_mask  = torch.where(any_real, safe_mask, chunk_mask.bool())
            chunk_cls   = chunk_cls * final_mask.unsqueeze(-1).float()

        # Signal cross-attention
        sig_q      = self.signal_emb(signal_ids).unsqueeze(1)
        key_pad    = (chunk_mask == 0)
        sig_ctx, _ = self.signal_cross_attn(
            query=sig_q, key=chunk_cls, value=chunk_cls,
            key_padding_mask=key_pad,
        )
        sig_ctx   = self.signal_norm(sig_q + sig_ctx)
        chunk_ctx = chunk_cls + sig_ctx
        chunk_ctx = chunk_ctx * chunk_mask.unsqueeze(-1).float()

        # Chunk MHA
        mha_out, _ = self.chunk_mha(
            query=chunk_ctx, key=chunk_ctx, value=chunk_ctx,
            key_padding_mask=key_pad,
        )
        chunk_ctx = self.mha_norm(chunk_ctx + self.mha_dropout(mha_out))
        chunk_ctx = chunk_ctx * chunk_mask.unsqueeze(-1).float()

        # BiLSTM
        lstm_out, _ = self.bilstm(chunk_ctx)
        lstm_out    = self.lstm_out_dropout(lstm_out)   # v2: extra dropout

        # Attention pooling
        scores   = self.attn_layer(lstm_out).squeeze(-1)
        scores   = scores.masked_fill(~chunk_mask.bool(), float("-inf"))
        weights  = F.softmax(scores, dim=1)
        doc_repr = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)

        logits = self.classifier(self.dropout(doc_repr))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels,
                                   label_smoothing=self.label_smoothing)

        class Out: pass
        o = Out(); o.loss = loss; o.logits = logits
        return o

    def resize_token_embeddings(self, n):
        self.bert.resize_token_embeddings(n)


# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════
class HierarchicalLegalQADataset(Dataset):
    def __init__(self, records, tokenizer,
                 max_chunk_len=256, max_chunks=2, with_signal=True):
        doc_groups = defaultdict(list)
        for r in records:
            doc_groups[r["doc_id"]].append(r)

        pad_ids  = torch.zeros(max_chunk_len, dtype=torch.long)
        pad_mask = torch.zeros(max_chunk_len, dtype=torch.long)

        self.samples = []
        for doc_id, qa_list in tqdm(doc_groups.items(),
                                    desc="  tokenising", leave=False):
            label    = int(qa_list[0]["label"])
            signals  = [r.get("signal", "NEUTRAL") for r in qa_list]
            dom_sig  = Counter(signals).most_common(1)[0][0]
            sig_idx  = SIGNAL_IDX.get(dom_sig, 2)

            # v2: max_chunks=2
            texts   = [format_input(r["question"], r["answer"],
                                    r["signal"], with_signal)
                       for r in qa_list][:max_chunks]
            n_real  = len(texts)

            all_ids, all_mask = [], []
            for text in texts:
                enc = tokenizer(text, max_length=max_chunk_len,
                                padding="max_length", truncation=True,
                                return_tensors="pt")
                all_ids.append(enc["input_ids"].squeeze(0))
                all_mask.append(enc["attention_mask"].squeeze(0))

            while len(all_ids) < max_chunks:
                all_ids.append(pad_ids.clone())
                all_mask.append(pad_mask.clone())

            self.samples.append({
                "chunk_input_ids"     : torch.stack(all_ids),
                "chunk_attention_mask": torch.stack(all_mask),
                "chunk_mask"          : torch.tensor(
                    [1]*n_real + [0]*(max_chunks - n_real), dtype=torch.long),
                "label"               : torch.tensor(label, dtype=torch.long),
                "signal_id"           : torch.tensor(sig_idx, dtype=torch.long),
                "doc_id"              : doc_id,
            })

        log.info(f"  Dataset ready : {len(self.samples)} docs (pre-tokenised)")

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    return {
        "chunk_input_ids"     : torch.stack([b["chunk_input_ids"]        for b in batch]),
        "chunk_attention_mask": torch.stack([b["chunk_attention_mask"]    for b in batch]),
        "chunk_mask"          : torch.stack([b["chunk_mask"]              for b in batch]),
        "label"               : torch.stack([b["label"]                   for b in batch]),
        "signal_id"           : torch.stack([b["signal_id"]               for b in batch]),
        "doc_id"              : [b["doc_id"] for b in batch],
    }


# ══════════════════════════════════════════════════════════════════════════════
# DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def load_data(path):
    recs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip(): recs.append(json.loads(line))
    return recs


def build_balanced_pool(records, max_docs=5000, seed=42):
    random.seed(seed)
    dg = defaultdict(list)
    for r in records: dg[r["doc_id"]].append(r)

    ids = list(dg.keys()); random.shuffle(ids)

    def dlabel(d):
        l = [int(r["label"]) for r in dg[d]]
        return 1 if l.count(1) >= l.count(0) else 0

    c0 = [d for d in ids if dlabel(d) == 0]
    c1 = [d for d in ids if dlabel(d) == 1]
    n  = min(len(c0), len(c1), max_docs // 2)
    bal = set(c0[:n] + c1[:n])

    pr = [r for r in records if r["doc_id"] in bal]
    pi = [d for d in ids     if d           in bal]
    log.info(f"  Balanced pool : {len(bal):,} docs  ({n} per class)  QA={len(pr):,}")
    return pr, pi


def split_train_val(pool_records, pool_ids, seed=42):
    n      = len(pool_ids)
    n_val  = max(1, int(round(n * 0.20)))
    n_tr   = n - n_val
    tr_ids = set(pool_ids[:n_tr]); va_ids = set(pool_ids[n_tr:])
    tr = [r for r in pool_records if r["doc_id"] in tr_ids]
    va = [r for r in pool_records if r["doc_id"] in va_ids]
    log.info(f"  Train : {n_tr} docs ({len(tr):,} QA)  |  Val : {n_val} docs ({len(va):,} QA)")
    return tr, va, n_tr, n_val


# ══════════════════════════════════════════════════════════════════════════════
# LLRD OPTIMISER  — all layers pre-registered, HEAD gets higher weight_decay
# ══════════════════════════════════════════════════════════════════════════════
def build_llrd_optimizer(model, lr_bert, lr_head, decay,
                         weight_decay, head_weight_decay):
    """
    v2 changes:
      - head param group now uses head_weight_decay (0.10) instead of
        the global weight_decay (0.05)
      - All other BERT / pooler groups use weight_decay (0.05)
      - Everything else unchanged: all layers pre-registered upfront so
        the LambdaLR scheduler group count never changes.
    """
    num_layers   = len(model.bert.encoder.layer)   # 12
    param_groups = []

    # Embeddings
    emb_lr     = lr_bert * (decay ** num_layers)
    emb_params = list(model.bert.embeddings.parameters())
    if emb_params:
        param_groups.append({
            "params"      : emb_params,
            "lr"          : emb_lr,
            "weight_decay": weight_decay,
            "name"        : "bert_emb",
        })

    # All 12 encoder layers (including frozen 0-9)
    for i, layer in enumerate(model.bert.encoder.layer):
        layer_lr     = lr_bert * (decay ** (num_layers - i))
        layer_params = list(layer.parameters())
        if layer_params:
            param_groups.append({
                "params"      : layer_params,
                "lr"          : layer_lr,
                "weight_decay": weight_decay,
                "name"        : f"bert_layer_{i}",
            })

    # Pooler
    pooler_p = (list(model.bert.pooler.parameters())
                if hasattr(model.bert, "pooler") else [])
    if pooler_p:
        param_groups.append({
            "params"      : pooler_p,
            "lr"          : lr_bert,
            "weight_decay": weight_decay,
            "name"        : "bert_pooler",
        })

    # Head — v2: uses head_weight_decay=0.10
    head_params = (
        list(model.signal_emb.parameters())
        + list(model.signal_cross_attn.parameters())
        + list(model.signal_norm.parameters())
        + list(model.chunk_mha.parameters())
        + list(model.mha_norm.parameters())
        + list(model.bilstm.parameters())
        + list(model.lstm_out_dropout.parameters())
        + list(model.attn_layer.parameters())
        + list(model.classifier.parameters())
    )
    param_groups.append({
        "params"      : head_params,
        "lr"          : lr_head,
        "weight_decay": head_weight_decay,   # v2: 0.10
        "name"        : "head",
    })

    param_groups = [g for g in param_groups if len(g["params"]) > 0]

    log.info(f"  LLRD param groups: {len(param_groups)}")
    for g in param_groups:
        n_total     = sum(p.numel() for p in g["params"])
        n_trainable = sum(p.numel() for p in g["params"] if p.requires_grad)
        log.info(f"    {g['name']:20s}  lr={g['lr']:.2e}  "
                 f"wd={g['weight_decay']:.3f}  "
                 f"total={n_total:,}  trainable={n_trainable:,}")

    return AdamW(param_groups)


# ══════════════════════════════════════════════════════════════════════════════
# DEFERRED CLASS REWEIGHTING
# ══════════════════════════════════════════════════════════════════════════════
def compute_class_weights(labels_list, device):
    cnt   = Counter(labels_list)
    n     = len(labels_list)
    n_cls = len(cnt)
    w = torch.tensor(
        [n / (n_cls * cnt.get(i, 1)) for i in range(n_cls)],
        dtype=torch.float, device=device,
    ).clamp(0.5, 2.0)
    log.info(f"  Class weights (active from ep {DEFERRED_RW_EPOCH}) : {w.cpu().tolist()}")
    return w


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN ONE EPOCH
# ══════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, scheduler, scaler,
                accum_steps, class_weights, epoch):
    model.train()
    total_loss = 0.0; n_correct = 0; n_total = 0
    optimizer.zero_grad()
    # v2: DEFERRED_RW_EPOCH=1 → class weights always active
    use_rw = (class_weights is not None) and (epoch >= DEFERRED_RW_EPOCH)

    pbar = tqdm(loader, desc=f"  Ep{epoch:02d} train", leave=False,
                dynamic_ncols=True)
    for step, batch in enumerate(pbar):
        ids  = batch["chunk_input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["chunk_attention_mask"].to(DEVICE, non_blocking=True)
        cmsk = batch["chunk_mask"].to(DEVICE, non_blocking=True)
        labs = batch["label"].to(DEVICE, non_blocking=True)
        sigs = batch["signal_id"].to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            out = model(ids, mask, cmsk, sigs, labs,
                        chunk_drop_prob=CHUNK_DROP_PROB)
            if use_rw:
                loss_raw = F.cross_entropy(
                    out.logits, labs,
                    weight=class_weights,
                    label_smoothing=LABEL_SMOOTHING,
                    reduction="mean",
                )
            else:
                loss_raw = out.loss
            loss = loss_raw / accum_steps

        scaler.scale(loss).backward()
        total_loss += loss_raw.item()
        preds       = torch.argmax(out.logits, dim=1)
        n_correct  += (preds == labs).sum().item()
        n_total    += len(labs)

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss_raw.item():.3f}",
                         acc=f"{n_correct/n_total:.3f}")

    return total_loss / len(loader), n_correct / n_total


# ══════════════════════════════════════════════════════════════════════════════
# EVALUATE
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(model, loader, class_weights=None, epoch=0):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    use_rw = (class_weights is not None) and (epoch >= DEFERRED_RW_EPOCH)

    with torch.no_grad():
        for batch in tqdm(loader, desc="  eval", leave=False,
                          dynamic_ncols=True):
            ids  = batch["chunk_input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["chunk_attention_mask"].to(DEVICE, non_blocking=True)
            cmsk = batch["chunk_mask"].to(DEVICE, non_blocking=True)
            labs = batch["label"].to(DEVICE, non_blocking=True)
            sigs = batch["signal_id"].to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):
                out = model(ids, mask, cmsk, sigs, labs, chunk_drop_prob=0.0)
                if use_rw:
                    loss_raw = F.cross_entropy(
                        out.logits, labs,
                        weight=class_weights,
                        label_smoothing=LABEL_SMOOTHING,
                    )
                else:
                    loss_raw = out.loss

            total_loss += loss_raw.item()
            probs = torch.softmax(out.logits.float(), dim=1).cpu().tolist()
            preds = torch.argmax(out.logits, dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labs.cpu().tolist())
            all_probs.extend([p[1] for p in probs])

    acc    = accuracy_score(all_labels, all_preds)
    f1     = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    prec   = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec    = recall_score(all_labels, all_preds, average="macro",    zero_division=0)
    f1_cls = f1_score(all_labels, all_preds, average=None,     zero_division=0)
    try:    auc = roc_auc_score(all_labels, all_probs)
    except: auc = 0.0
    try:    mcc = matthews_corrcoef(all_labels, all_preds)
    except: mcc = 0.0
    try:    kap = cohen_kappa_score(all_labels, all_preds)
    except: kap = 0.0

    dist = Counter(all_preds)
    if len(dist) < 2:
        log.warning(f"  ⚠️  Class collapse: {dict(dist)}")

    return {
        "loss"    : total_loss / len(loader),
        "acc"     : acc,  "f1"  : f1,
        "prec"    : prec, "rec" : rec,
        "auc"     : auc,  "mcc" : mcc, "kappa": kap,
        "f1_rej"  : float(f1_cls[0]) if len(f1_cls) > 0 else 0.0,
        "f1_acc"  : float(f1_cls[1]) if len(f1_cls) > 1 else 0.0,
        "preds"   : all_preds, "labels": all_labels, "probs": all_probs,
        "pred_dist": dict(dist),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ADAPTIVE HYPERPARAMETER CONTROLLER  (v2: auto-unfreeze DISABLED)
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveHPController:
    """
    v2 changes:
      - unfreeze_bert_from() call is REMOVED from the underfitting branch.
        Unfreezing was causing more params to receive gradients and worsening
        the already-severe overfitting gap.
      - Only dropout bump and weight_decay bump remain active.
      - LR bump for head also disabled (LR is already very small).
    """
    def __init__(self):
        self.overfit_streak = 0
        self.dropout_bumped = False
        self.wd_bumped      = False

    def step(self, epoch, train_loss, val_loss, val_f1, model, optimizer):
        actions = []

        # ── Overfitting: bump dropout then WD ─────────────────────────────────
        if val_loss - train_loss > OVERFIT_GAP_THRESH:
            self.overfit_streak += 1
        else:
            self.overfit_streak  = 0

        if self.overfit_streak >= OVERFIT_PATIENCE:
            if not self.dropout_bumped:
                for m in model.modules():
                    if isinstance(m, nn.Dropout):
                        m.p = min(m.p + 0.05, 0.55)   # cap at 0.55
                self.dropout_bumped = True
                dp = [m.p for m in model.modules() if isinstance(m, nn.Dropout)]
                actions.append(f"dropout→{dp[0]:.2f}")
            elif not self.wd_bumped:
                for pg in optimizer.param_groups:
                    pg["weight_decay"] = min(pg["weight_decay"] * 1.5, 0.15)
                self.wd_bumped = True
                actions.append("weight_decay bumped")

        # ── Underfitting: log only — NO unfreeze in v2 ────────────────────────
        if epoch >= 8 and val_f1 < UNDERFIT_F1_THRESH:
            log.info(f"  [AdaptiveHP ep{epoch}] Underfitting detected "
                     f"(val_f1={val_f1:.3f}) — auto-unfreeze DISABLED in v2")

        if actions:
            log.info(f"  [AdaptiveHP ep{epoch}] Actions: {' | '.join(actions)}")
        return actions


# ══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ══════════════════════════════════════════════════════════════════════════════
def save_plots(history, labels, preds, swa_start):
    ep         = [h["epoch"]      for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss   = [h["val_loss"]   for h in history]
    val_f1     = [h["val_f1"]     for h in history]
    val_acc    = [h["val_acc"]    for h in history]
    val_auc    = [h["val_auc"]    for h in history]
    val_mcc    = [h["val_mcc"]    for h in history]
    train_acc  = [h["train_acc"]  for h in history]
    f1_rej     = [h["val_f1_rej"] for h in history]
    f1_acc_cls = [h["val_f1_acc"] for h in history]

    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle(
        "InLegalBERT v2 — Anti-Overfit (dropout↑, LR↓, WD↑, BERT frozen×10, SWA@10)",
        fontsize=13, fontweight="bold")

    ax = axes[0, 0]
    ax.plot(ep, train_loss, "b-o", ms=4, label="Train Loss")
    ax.plot(ep, val_loss,   "r-o", ms=4, label="Val Loss")
    if swa_start <= max(ep):
        ax.axvline(swa_start, color="orange", linestyle="--", alpha=0.7,
                   label=f"SWA start (ep{swa_start})")
    ax.fill_between(ep,
                    [abs(v - t) for v, t in zip(val_loss, train_loss)],
                    alpha=0.15, color="red", label="Overfit gap")
    ax.set_title("Loss Curve"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(ep, val_f1,    "g-s", ms=4, label="Val Macro-F1")
    ax.plot(ep, train_acc, "b-s", ms=4, label="Train Acc")
    ax.plot(ep, val_acc,   "r-s", ms=4, label="Val Acc")
    ax.axhline(SOTA_F1, color="purple", linestyle="--",
               label=f"SOTA F1={SOTA_F1}")
    ax.set_title("F1 / Accuracy"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)

    ax = axes[0, 2]
    ax.plot(ep, val_auc, "m-^", ms=4, label="Val AUC-ROC")
    ax.plot(ep, val_mcc, "c-^", ms=4, label="Val MCC")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_title("AUC & MCC"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(ep, f1_rej,     "r-o", ms=4, label="F1 REJECTED")
    ax.plot(ep, f1_acc_cls, "g-o", ms=4, label="F1 ACCEPTED")
    ax.set_title("Per-class F1"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)

    ax = axes[1, 1]
    gap = [v - t for v, t in zip(val_loss, train_loss)]
    ax.plot(ep, gap, "k-o", ms=4)
    ax.axhline(OVERFIT_GAP_THRESH, color="red", linestyle="--",
               label=f"Overfit thresh={OVERFIT_GAP_THRESH}")
    ax.axhline(0, color="gray", linestyle=":")
    ax.fill_between(ep, gap, 0,
                    where=[g > 0 for g in gap],
                    alpha=0.2, color="red",  label="Overfitting")
    ax.fill_between(ep, gap, 0,
                    where=[g <= 0 for g in gap],
                    alpha=0.2, color="blue", label="Underfitting")
    ax.set_title("Train-Val Loss Gap"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 2]
    cm = confusion_matrix(labels, preds)
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["REJECTED", "ACCEPTED"])
    ax.set_yticklabels(["REJECTED", "ACCEPTED"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix — Best Epoch")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i][j]), ha="center", va="center",
                    fontsize=12, fontweight="bold",
                    color="white" if cm[i][j] > cm.max() / 2 else "black")

    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/full_analysis_v2.png", dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"  Plots → {PLOT_DIR}/full_analysis_v2.png")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":

    # ── Data ──────────────────────────────────────────────────────────────────
    log.info("=" * 60 + "\n  LOADING DATA\n" + "=" * 60)
    records = load_data(INPUT_PATH)
    log.info(f"  QA pairs : {len(records):,}  |  "
             f"Docs : {len(set(r['doc_id'] for r in records)):,}")
    pool_records, pool_ids = build_balanced_pool(
        records, max_docs=MAX_TOTAL_DOCS, seed=SEED)
    train_records, val_records, n_tr, n_va = split_train_val(
        pool_records, pool_ids, seed=SEED)

    # ── Tokeniser ─────────────────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(INLEGAL_MODEL_ID)
    if WITH_SIGNAL:
        tokenizer.add_tokens(["[FP]", "[FR]", "[N]"])
        log.info(f"  Vocab size : {len(tokenizer):,}")

    # ── Datasets ──────────────────────────────────────────────────────────────
    log.info("=" * 60 + "\n  PRE-TOKENISING\n" + "=" * 60)
    train_ds = HierarchicalLegalQADataset(
        train_records, tokenizer, MAX_CHUNK_LEN, MAX_CHUNKS, WITH_SIGNAL)
    val_ds   = HierarchicalLegalQADataset(
        val_records,   tokenizer, MAX_CHUNK_LEN, MAX_CHUNKS, WITH_SIGNAL)

    doc_labels_train = [s["label"].item() for s in train_ds.samples]
    cnt  = Counter(doc_labels_train)
    n0, n1 = cnt.get(0, 1), cnt.get(1, 1)
    log.info(f"  Train class dist → REJECTED={n0}  ACCEPTED={n1}")

    # Weighted sampler for balanced mini-batches
    w = torch.tensor([
        len(doc_labels_train) / (2.0 * n0) if l == 0
        else len(doc_labels_train) / (2.0 * n1)
        for l in doc_labels_train
    ], dtype=torch.float)
    sampler       = WeightedRandomSampler(w, len(w), replacement=True)
    # v2: class weights active from ep 1 (DEFERRED_RW_EPOCH=1)
    class_weights = compute_class_weights(doc_labels_train, DEVICE)

    # v2: BATCH_SIZE=4, ACCUM_STEPS=4 → effective batch=16
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler,
        collate_fn=collate_fn, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
        collate_fn=collate_fn, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    log.info("=" * 60 + "\n  BUILDING MODEL\n" + "=" * 60)
    model = HierarchicalInLegalBERT(
        model_id=INLEGAL_MODEL_ID, num_labels=2,
        dropout=DROPOUT,                   # 0.4
        lstm_hidden=LSTM_HIDDEN,           # 128
        lstm_layers=LSTM_LAYERS,           # 1
        lstm_dropout=LSTM_DROPOUT,         # 0.3
        mha_heads=MHA_HEADS,               # 4
        mha_dropout=MHA_DROPOUT,           # 0.3
        label_smoothing=LABEL_SMOOTHING,   # 0.15
        freeze_bert_layers=FREEZE_BERT_LAYERS,  # 10
    )
    if WITH_SIGNAL:
        model.resize_token_embeddings(len(tokenizer))
    model = model.to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    log.info(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    # ── LLRD Optimizer (all layers pre-registered, head WD=0.10) ──────────────
    optimizer = build_llrd_optimizer(
        model, LR_BERT, LR_HEAD, LLRD_DECAY,
        WEIGHT_DECAY, HEAD_WEIGHT_DECAY)

    steps_per_epoch = (len(train_loader) + ACCUM_STEPS - 1) // ACCUM_STEPS
    total_steps     = steps_per_epoch * MAX_EPOCHS
    warmup_steps    = int(total_steps * WARMUP_RATIO)
    log.info(f"  Steps/ep={steps_per_epoch}  total={total_steps}  warmup={warmup_steps}")

    scheduler = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps)
    scaler    = GradScaler(enabled=USE_AMP)

    # ── SWA  (v2: starts at ep 10, LR=1e-6) ──────────────────────────────────
    swa_model     = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR,
                          anneal_epochs=5, anneal_strategy="cos")
    swa_active    = False

    # ── Adaptive HP ───────────────────────────────────────────────────────────
    ahp = AdaptiveHPController()

    # ── Resume ────────────────────────────────────────────────────────────────
    start_epoch  = 1
    best_f1      = 0.0
    best_epoch   = 0
    best_metrics = {}
    no_improve   = 0
    history      = []

    if os.path.exists(CKPT_ROLL):
        try:
            ck = torch.load(CKPT_ROLL, map_location=DEVICE)
            model.load_state_dict(ck["model_state"])
            optimizer.load_state_dict(ck["optimizer_state"])
            scheduler.load_state_dict(ck["scheduler_state"])
            scaler.load_state_dict(ck["scaler_state"])
            start_epoch  = ck["epoch"] + 1
            best_f1      = ck["best_f1"]
            best_epoch   = ck["best_epoch"]
            best_metrics = ck["best_metrics"]
            no_improve   = ck["no_improve"]
            history      = ck["history"]
            if ck.get("swa_state"):
                swa_model.load_state_dict(ck["swa_state"])
            log.info(f"  ▶ RESUMED from epoch {ck['epoch']} "
                     f"(best F1={best_f1:.4f})")
        except Exception as e:
            log.warning(f"  ⚠️  Could not load checkpoint: {e} — starting fresh")

    # ── CSV ───────────────────────────────────────────────────────────────────
    csv_exists = os.path.exists(CSV_PATH) and start_epoch > 1
    csv_file   = open(CSV_PATH, "a" if csv_exists else "w", newline="")
    csv_writer = csv.writer(csv_file)
    if not csv_exists:
        csv_writer.writerow([
            "epoch","train_loss","train_acc",
            "val_loss","val_acc","val_f1","val_prec","val_rec",
            "val_auc","val_mcc","val_kappa",
            "val_f1_rej","val_f1_acc","overfit_gap",
            "swa_active","epoch_secs","adaptive_actions",
        ])

    # ── Training loop ─────────────────────────────────────────────────────────
    log.info("=" * 60)
    log.info(f"  TRAINING v2 — {MAX_EPOCHS} epochs | {n_tr} train | {n_va} val")
    log.info("=" * 60)

    start_time = datetime.now()

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        ep_start = datetime.now()

        # v2: SWA starts at epoch 10
        if epoch >= SWA_START and not swa_active:
            swa_active = True
            log.info(f"  🔄  SWA activated at epoch {epoch}")

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, scaler,
            ACCUM_STEPS, class_weights, epoch)

        val_m = evaluate(model, val_loader, class_weights, epoch)

        if swa_active:
            swa_model.update_parameters(model)
            swa_scheduler.step()

        actions = ahp.step(
            epoch, train_loss, val_m["loss"], val_m["f1"], model, optimizer)

        ep_secs  = (datetime.now() - ep_start).total_seconds()
        done_min = (datetime.now() - start_time).total_seconds() / 60
        eta_min  = ep_secs * (MAX_EPOCHS - epoch) / 60
        gap      = val_m["loss"] - train_loss

        log.info(
            f"  Ep {epoch:02d}/{MAX_EPOCHS} | "
            f"TrLoss={train_loss:.4f} TrAcc={train_acc:.4f} | "
            f"VaLoss={val_m['loss']:.4f} VaAcc={val_m['acc']:.4f} "
            f"VaF1={val_m['f1']:.4f} | "
            f"AUC={val_m['auc']:.4f} MCC={val_m['mcc']:.4f} "
            f"κ={val_m['kappa']:.4f} | "
            f"F1[REJ={val_m['f1_rej']:.3f} ACC={val_m['f1_acc']:.3f}] | "
            f"Gap={gap:+.4f} SWA={'✓' if swa_active else '✗'} | "
            f"{ep_secs:.0f}s elapsed={done_min:.0f}m ETA≈{eta_min:.0f}m"
        )

        history.append({
            "epoch"     : epoch,
            "train_loss": round(train_loss,      4),
            "train_acc" : round(train_acc,       4),
            "val_loss"  : round(val_m["loss"],   4),
            "val_f1"    : round(val_m["f1"],     4),
            "val_acc"   : round(val_m["acc"],    4),
            "val_auc"   : round(val_m["auc"],    4),
            "val_mcc"   : round(val_m["mcc"],    4),
            "val_f1_rej": round(val_m["f1_rej"], 4),
            "val_f1_acc": round(val_m["f1_acc"], 4),
        })
        csv_writer.writerow([
            epoch, round(train_loss, 4), round(train_acc, 4),
            round(val_m["loss"],  4), round(val_m["acc"],   4),
            round(val_m["f1"],    4), round(val_m["prec"],  4),
            round(val_m["rec"],   4), round(val_m["auc"],   4),
            round(val_m["mcc"],   4), round(val_m["kappa"], 4),
            round(val_m["f1_rej"], 4), round(val_m["f1_acc"], 4),
            round(gap, 4), int(swa_active), round(ep_secs, 1),
            "|".join(actions),
        ])
        csv_file.flush()

        if val_m["f1"] > best_f1:
            best_f1 = val_m["f1"]; best_epoch = epoch
            best_metrics = val_m; no_improve = 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "best_f1": best_f1, "val_acc": val_m["acc"],
                "val_auc": val_m["auc"], "val_mcc": val_m["mcc"],
            }, CKPT_BEST)
            log.info(f"  ✅  New best F1={best_f1:.4f} → {CKPT_BEST}")
        else:
            no_improve += 1
            log.info(f"  No improve {no_improve}/{EARLY_STOP_PAT} "
                     f"(best F1={best_f1:.4f} @ ep {best_epoch})")

        torch.save({
            "epoch"          : epoch,
            "model_state"    : model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state"   : scaler.state_dict(),
            "swa_state"      : swa_model.state_dict() if swa_active else None,
            "best_f1"        : best_f1,
            "best_epoch"     : best_epoch,
            "best_metrics"   : best_metrics,
            "no_improve"     : no_improve,
            "history"        : history,
        }, CKPT_ROLL)

        if no_improve >= EARLY_STOP_PAT:
            log.info(f"  ⏹  Early stopping at epoch {epoch}")
            break

    csv_file.close()

    # ── SWA final BN update ───────────────────────────────────────────────────
    if swa_active:
        log.info("  🔄  Updating SWA BatchNorm statistics ...")
        update_bn(train_loader, swa_model, device=DEVICE)
        swa_val = evaluate(swa_model, val_loader, class_weights, MAX_EPOCHS)
        log.info(f"  SWA model → F1={swa_val['f1']:.4f}  "
                 f"Acc={swa_val['acc']:.4f}  AUC={swa_val['auc']:.4f}")
        if swa_val["f1"] > best_f1:
            torch.save({"model_state": swa_model.state_dict(),
                        "source": "SWA", "f1": swa_val["f1"]},
                       f"{OUTPUT_DIR}/swa_best_model.pt")
            log.info("  ✅  SWA model is best → saved")

    # ── Final report ──────────────────────────────────────────────────────────
    total_mins = (datetime.now() - start_time).total_seconds() / 60
    report = classification_report(
        best_metrics["labels"], best_metrics["preds"],
        target_names=["REJECTED", "ACCEPTED"], digits=4,
    )
    log.info("\n" + "=" * 60)
    log.info(f"  FINAL RESULTS v2  (best epoch = {best_epoch})")
    log.info("=" * 60)
    log.info(f"  Val Acc   : {best_metrics['acc']:.4f}   SOTA={SOTA_ACC}")
    log.info(f"  Val F1    : {best_metrics['f1']:.4f}   SOTA={SOTA_F1}")
    log.info(f"  Val AUC   : {best_metrics['auc']:.4f}")
    log.info(f"  Val MCC   : {best_metrics['mcc']:.4f}")
    log.info(f"  Val κ     : {best_metrics['kappa']:.4f}")
    log.info(f"  F1 REJ    : {best_metrics['f1_rej']:.4f}")
    log.info(f"  F1 ACC    : {best_metrics['f1_acc']:.4f}")
    log.info(f"  Runtime   : {total_mins:.1f} min")
    log.info(f"\n{report}")

    save_plots(history, best_metrics["labels"],
               best_metrics["preds"], SWA_START)

    log.info(f"  Best model  → {CKPT_BEST}")
    log.info(f"  Last ckpt   → {CKPT_ROLL}  (resume-safe)")
    log.info(f"  CSV         → {CSV_PATH}")
    log.info(f"  Plots       → {PLOT_DIR}/full_analysis_v2.png")
    log.info(f"  Log         → {log_file}")
    log.info("  ✅  Done.")

16:45:02 | Device        : cuda  |  AMP: True
16:45:02 | Architecture  : InLegalBERT → SignalCrossAttn → MHA(4h) → BiLSTM(128h,1L) → AttnPool → Linear
16:45:02 | Epochs        : 50  patience=15  SWA from ep 10
16:45:02 | LR BERT/HEAD  : 5e-06/5e-06  LLRD=0.95  WD=0.05  HeadWD=0.1
16:45:02 | Dropout       : main=0.4  lstm=0.3  mha=0.3
16:45:02 | MAX_CHUNKS    : 2  CHUNK_DROP=0.35  LABEL_SMOOTH=0.15
16:45:02 | FREEZE_BERT   : 10 layers  (auto-unfreeze DISABLED)
16:45:02 | DEFERRED_RW   : ep 1 (effective from start)
16:45:02 | v2 changes    : dropout↑ | complexity↓ | LR↓ | WD↑ | freeze↑ | SWA early | class-weights early
16:45:02 | ============================================================
  LOADING DATA
16:45:03 |   QA pairs : 45,329  |  Docs : 5,421
16:45:03 |   Balanced pool : 4,738 docs  (2369 per class)  QA=39,604
16:45:03 |   Train : 3790 docs (31,722 QA)  |  Val : 948 docs (7,882 QA)
16:45:03 | HTTP Request: HEAD https://huggingface.co/law-ai/InCaseLawBERT/resolve/main/config.json

  tokenising:   0%|          | 0/3790 [00:00<?, ?it/s]

16:45:08 |   Dataset ready : 3790 docs (pre-tokenised)


  tokenising:   0%|          | 0/948 [00:00<?, ?it/s]

16:45:10 |   Dataset ready : 948 docs (pre-tokenised)
16:45:10 |   Train class dist → REJECTED=2130  ACCEPTED=1660
16:45:10 |   Class weights (active from ep 1) : [0.8896713852882385, 1.141566276550293]
16:45:10 | ============================================================
  BUILDING MODEL
16:45:10 | HTTP Request: HEAD https://huggingface.co/law-ai/InCaseLawBERT/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
16:45:10 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/law-ai/InCaseLawBERT/7f2c2a0c1ff4149e8c4a8c79ee9f24757ad5dacd/config.json "HTTP/1.1 200 OK"
16:45:11 | HTTP Request: HEAD https://huggingface.co/law-ai/InCaseLawBERT/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InCaseLawBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
16:45:11 | HTTP Request: GET https://huggingface.co/api/models/law-ai/InCaseLawBERT "H

  Ep01 train:   0%|                                                                             | 0/948 [00:00…

16:45:13 | HTTP Request: HEAD https://huggingface.co/law-ai/InCaseLawBERT/resolve/refs%2Fpr%2F2/model.safetensors "HTTP/1.1 302 Found"


In [1]:
# model_single_run_v2_fast.py
# ─────────────────────────────────────────────────────────────────────────────
# Same as model_single_run_v2.py but with Jupyter/speed fixes:
#
# SPEED FIXES vs v2:
#   1. num_workers=0  (eliminates Jupyter DataLoader multiprocessing stall)
#   2. persistent_workers / prefetch_factor removed (only valid when workers>0)
#   3. BATCH_SIZE 4 → 16  |  ACCUM_STEPS 4 → 1  (same effective batch=16,
#      but ~4x fewer optimizer steps per epoch → ~4x faster per epoch)
#   4. if __name__ == "__main__" guard REMOVED (Jupyter safe)
#   5. logging.basicConfig replaced with force-reset handler (works in Jupyter
#      even if root logger was already configured by a previous cell)
#   6. MAX_TOTAL_DOCS cap added as a quick-test knob (set to 5000 for full run)
# ─────────────────────────────────────────────────────────────────────────────

import os, gc, json, random, logging, warnings, csv, math
from copy import deepcopy
from datetime import datetime
from collections import defaultdict, Counter
from typing import Optional, List, Dict

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
INPUT_PATH = "Track_A_qa_judgment_flat_OLLAMA.jsonl"
OUTPUT_DIR = "single_run_results_INcaselawbert"
LOG_DIR    = f"{OUTPUT_DIR}/logs"
PLOT_DIR   = f"{OUTPUT_DIR}/plots"
CKPT_ROLL  = f"{OUTPUT_DIR}/checkpoint_last.pt"
CKPT_BEST  = f"{OUTPUT_DIR}/best_model.pt"
CSV_PATH   = f"{OUTPUT_DIR}/epoch_results.csv"

INLEGAL_MODEL_ID = "law-ai/InCaseLawBERT"

MAX_TOTAL_DOCS = 5000
MAX_EPOCHS     = 50
EARLY_STOP_PAT = 15

# ── SPEED FIX: larger batch, no accumulation ─────────────────────────────────
BATCH_SIZE  = 16   # was 4  → fewer steps per epoch, same GPU memory via AMP
ACCUM_STEPS = 1    # was 4  → effective batch = 16 (same), but 4x fewer steps

# ── Learning rates & regularisation ──────────────────────────────────────────
LR_BERT           = 5e-6
LR_HEAD           = 5e-6
LLRD_DECAY        = 0.95
WARMUP_RATIO      = 0.06
WEIGHT_DECAY      = 0.05
HEAD_WEIGHT_DECAY = 0.10

# ── Architecture ──────────────────────────────────────────────────────────────
MAX_CHUNK_LEN      = 256
MAX_CHUNKS         = 2
LSTM_HIDDEN        = 128
LSTM_LAYERS        = 1
LSTM_DROPOUT       = 0.3
MHA_HEADS          = 4
MHA_DROPOUT        = 0.3
DROPOUT            = 0.4
FREEZE_BERT_LAYERS = 10

# ── Training tricks ───────────────────────────────────────────────────────────
WITH_SIGNAL       = True
LABEL_SMOOTHING   = 0.15
DEFERRED_RW_EPOCH = 1
CHUNK_DROP_PROB   = 0.35
SWA_START         = 10
SWA_LR            = 1e-6

OVERFIT_GAP_THRESH = 0.15
OVERFIT_PATIENCE   = 3
UNDERFIT_F1_THRESH = 0.55

SEED     = 42
SOTA_F1  = 0.8131
SOTA_ACC = 0.78
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP  = DEVICE == "cuda"

for d in [OUTPUT_DIR, LOG_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING  — force-reset so it works even after kernel re-use in Jupyter
# ══════════════════════════════════════════════════════════════════════════════
run_id   = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"{LOG_DIR}/run_{run_id}.log"

# Force-reset: removes any handlers attached by a previous cell / basicConfig
root_logger = logging.getLogger()
root_logger.handlers.clear()
root_logger.setLevel(logging.INFO)
_fmt = logging.Formatter("%(asctime)s | %(message)s", datefmt="%H:%M:%S")
_fh  = logging.FileHandler(log_file);  _fh.setFormatter(_fmt);  root_logger.addHandler(_fh)
_sh  = logging.StreamHandler();        _sh.setFormatter(_fmt);  root_logger.addHandler(_sh)
log  = root_logger

log.info(f"Device        : {DEVICE}  |  AMP: {USE_AMP}")
log.info(f"SPEED FIX     : num_workers=0 | BATCH={BATCH_SIZE} | ACCUM={ACCUM_STEPS} (eff batch=16)")
log.info(f"Architecture  : InCaseLawBERT → SignalCrossAttn → MHA({MHA_HEADS}h) → BiLSTM({LSTM_HIDDEN}h,{LSTM_LAYERS}L) → AttnPool → Linear")
log.info(f"Epochs        : {MAX_EPOCHS}  patience={EARLY_STOP_PAT}  SWA from ep {SWA_START}")
log.info(f"LR BERT/HEAD  : {LR_BERT}/{LR_HEAD}  LLRD={LLRD_DECAY}  WD={WEIGHT_DECAY}  HeadWD={HEAD_WEIGHT_DECAY}")
log.info(f"Dropout       : main={DROPOUT}  lstm={LSTM_DROPOUT}  mha={MHA_DROPOUT}")
log.info(f"MAX_CHUNKS    : {MAX_CHUNKS}  CHUNK_DROP={CHUNK_DROP_PROB}  LABEL_SMOOTH={LABEL_SMOOTHING}")
log.info(f"FREEZE_BERT   : {FREEZE_BERT_LAYERS} layers  (auto-unfreeze DISABLED)")


# ══════════════════════════════════════════════════════════════════════════════
# SEED
# ══════════════════════════════════════════════════════════════════════════════
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(SEED)
torch.backends.cudnn.enabled   = True
torch.backends.cudnn.benchmark = True


# ══════════════════════════════════════════════════════════════════════════════
# SIGNAL TOKENS
# ══════════════════════════════════════════════════════════════════════════════
SIGNAL_MAP = {
    "FAVORS_PETITIONER": "[FP]",
    "FAVORS_RESPONDENT": "[FR]",
    "NEUTRAL"          : "[N]",
}
SIGNAL_IDX = {"FAVORS_PETITIONER": 0, "FAVORS_RESPONDENT": 1, "NEUTRAL": 2}

def format_input(question: str, answer: str, signal: str,
                 with_signal: bool = True) -> str:
    sig = SIGNAL_MAP.get(signal, "[N]") if with_signal else ""
    return f"Q: {question.strip()} A: {answer.strip()} {sig}".strip()


# ══════════════════════════════════════════════════════════════════════════════
# MODEL  (identical to v2)
# ══════════════════════════════════════════════════════════════════════════════
class HierarchicalInLegalBERT(nn.Module):
    def __init__(self, model_id, num_labels=2, dropout=0.4,
                 lstm_hidden=128, lstm_layers=1, lstm_dropout=0.3,
                 mha_heads=4, mha_dropout=0.3,
                 label_smoothing=0.15, freeze_bert_layers=10):
        super().__init__()
        self.label_smoothing    = label_smoothing
        self.freeze_bert_layers = freeze_bert_layers

        self.bert = AutoModel.from_pretrained(model_id)
        D = self.bert.config.hidden_size
        self._freeze_bert(freeze_bert_layers)

        self.signal_emb        = nn.Embedding(3, D)
        nn.init.normal_(self.signal_emb.weight, std=0.02)

        self.signal_cross_attn = nn.MultiheadAttention(D, mha_heads, dropout=mha_dropout, batch_first=True)
        self.signal_norm       = nn.LayerNorm(D)
        self.chunk_mha         = nn.MultiheadAttention(D, mha_heads, dropout=mha_dropout, batch_first=True)
        self.mha_norm          = nn.LayerNorm(D)
        self.mha_dropout       = nn.Dropout(mha_dropout)

        bilstm_out = lstm_hidden * 2
        self.bilstm = nn.LSTM(
            input_size=D, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True, bidirectional=True,
            dropout=lstm_dropout if lstm_layers > 1 else 0.0,
        )
        self.lstm_out_dropout = nn.Dropout(lstm_dropout)

        self.attn_layer = nn.Linear(bilstm_out, 1)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(bilstm_out, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def _freeze_bert(self, n):
        for p in self.bert.embeddings.parameters(): p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters(): p.requires_grad = (i >= n)

    def unfreeze_bert_from(self, n):
        self._freeze_bert(n); self.freeze_bert_layers = n
        log.info(f"  [unfreeze] layer≥{n} → {sum(p.numel() for p in self.parameters() if p.requires_grad):,} trainable")

    def encode_chunks(self, ids, mask, cmsk):
        B, N, L = ids.shape
        out = self.bert(input_ids=ids.view(B*N, L), attention_mask=mask.view(B*N, L))
        return out.last_hidden_state[:, 0, :].view(B, N, -1) * cmsk.unsqueeze(-1).float()

    def forward(self, chunk_input_ids, chunk_attention_mask, chunk_mask,
                signal_ids, labels=None, chunk_drop_prob=0.0):

        x = self.encode_chunks(chunk_input_ids, chunk_attention_mask, chunk_mask)

        if chunk_drop_prob > 0.0 and self.training:
            dm  = torch.rand(x.shape[:2], device=x.device) > chunk_drop_prob
            sm  = chunk_mask.bool() & dm
            sm  = torch.where(sm.any(1, keepdim=True), sm, chunk_mask.bool())
            x   = x * sm.unsqueeze(-1).float()

        kp      = (chunk_mask == 0)
        sq      = self.signal_emb(signal_ids).unsqueeze(1)
        sc, _   = self.signal_cross_attn(sq, x, x, key_padding_mask=kp)
        sc      = self.signal_norm(sq + sc)
        x       = (x + sc) * chunk_mask.unsqueeze(-1).float()

        mx, _   = self.chunk_mha(x, x, x, key_padding_mask=kp)
        x       = self.mha_norm(x + self.mha_dropout(mx)) * chunk_mask.unsqueeze(-1).float()

        lo, _   = self.bilstm(x)
        lo      = self.lstm_out_dropout(lo)

        sc2     = self.attn_layer(lo).squeeze(-1).masked_fill(~chunk_mask.bool(), float("-inf"))
        w       = F.softmax(sc2, dim=1)
        dr      = (lo * w.unsqueeze(-1)).sum(1)

        logits  = self.classifier(self.dropout(dr))
        loss    = F.cross_entropy(logits, labels, label_smoothing=self.label_smoothing) if labels is not None else None

        class O: pass
        o = O(); o.loss = loss; o.logits = logits
        return o

    def resize_token_embeddings(self, n):
        self.bert.resize_token_embeddings(n)


# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════
class HierarchicalLegalQADataset(Dataset):
    def __init__(self, records: List[Dict], tokenizer,
                 max_chunk_len=256, max_chunks=2, with_signal=True):
        doc_groups: Dict[str, List[Dict]] = defaultdict(list)
        for r in records:
            doc_groups[r["doc_id"]].append(r)

        pad_ids  = torch.zeros(max_chunk_len, dtype=torch.long)
        pad_mask = torch.zeros(max_chunk_len, dtype=torch.long)

        self.samples = []
        for doc_id, qa_list in tqdm(doc_groups.items(), desc="  tokenising", leave=False):
            label   = int(qa_list[0]["label"])
            signals = [r.get("signal", "NEUTRAL") for r in qa_list]
            sig_idx = SIGNAL_IDX.get(Counter(signals).most_common(1)[0][0], 2)

            texts  = [format_input(r["question"], r["answer"],
                                   r.get("signal", "NEUTRAL"), with_signal)
                      for r in qa_list][:max_chunks]
            n_real = len(texts)

            all_ids, all_mask = [], []
            for text in texts:
                enc = tokenizer(text, max_length=max_chunk_len,
                                padding="max_length", truncation=True,
                                return_tensors="pt")
                all_ids.append(enc["input_ids"].squeeze(0))
                all_mask.append(enc["attention_mask"].squeeze(0))
            while len(all_ids) < max_chunks:
                all_ids.append(pad_ids.clone())
                all_mask.append(pad_mask.clone())

            self.samples.append({
                "chunk_input_ids"     : torch.stack(all_ids),
                "chunk_attention_mask": torch.stack(all_mask),
                "chunk_mask"          : torch.tensor([1]*n_real + [0]*(max_chunks-n_real), dtype=torch.long),
                "label"               : torch.tensor(label,   dtype=torch.long),
                "signal_id"           : torch.tensor(sig_idx, dtype=torch.long),
                "doc_id"              : doc_id,
            })

        log.info(f"  Dataset ready : {len(self.samples)} docs (pre-tokenised)")

    def __len__(self):        return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    return {
        "chunk_input_ids"     : torch.stack([b["chunk_input_ids"]      for b in batch]),
        "chunk_attention_mask": torch.stack([b["chunk_attention_mask"]  for b in batch]),
        "chunk_mask"          : torch.stack([b["chunk_mask"]            for b in batch]),
        "label"               : torch.stack([b["label"]                 for b in batch]),
        "signal_id"           : torch.stack([b["signal_id"]             for b in batch]),
        "doc_id"              : [b["doc_id"] for b in batch],
    }


# ══════════════════════════════════════════════════════════════════════════════
# DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def load_data(path: str) -> List[Dict]:
    recs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip(): recs.append(json.loads(line))
    return recs


def build_balanced_pool(records: List[Dict], max_docs=5000, seed=42):
    random.seed(seed)
    dg: Dict[str, List[Dict]] = defaultdict(list)
    for r in records: dg[r["doc_id"]].append(r)
    ids = list(dg.keys()); random.shuffle(ids)
    def dlabel(d):
        l = [int(r["label"]) for r in dg[d]]
        return 1 if l.count(1) >= l.count(0) else 0
    c0 = [d for d in ids if dlabel(d)==0]
    c1 = [d for d in ids if dlabel(d)==1]
    n  = min(len(c0), len(c1), max_docs//2)
    bal = set(c0[:n] + c1[:n])
    pr = [r for r in records if r["doc_id"] in bal]
    pi = [d for d in ids     if d in bal]
    log.info(f"  Balanced pool : {len(bal):,} docs  ({n}/class)  QA={len(pr):,}")
    return pr, pi


def split_train_val(pool_records: List[Dict], pool_ids: List[str], seed=42):
    n    = len(pool_ids)
    nv   = max(1, int(round(n * 0.20)))
    nt   = n - nv
    tr_ids = set(pool_ids[:nt]); va_ids = set(pool_ids[nt:])
    tr = [r for r in pool_records if r["doc_id"] in tr_ids]
    va = [r for r in pool_records if r["doc_id"] in va_ids]
    log.info(f"  Train : {nt} docs ({len(tr):,} QA)  |  Val : {nv} docs ({len(va):,} QA)")
    return tr, va, nt, nv


# ══════════════════════════════════════════════════════════════════════════════
# LLRD OPTIMIZER
# ══════════════════════════════════════════════════════════════════════════════
def build_llrd_optimizer(model, lr_bert, lr_head, decay, weight_decay, head_wd):
    nl = len(model.bert.encoder.layer)
    pgs = []

    ep = list(model.bert.embeddings.parameters())
    if ep: pgs.append({"params": ep, "lr": lr_bert*(decay**nl), "weight_decay": weight_decay, "name": "bert_emb"})

    for i, layer in enumerate(model.bert.encoder.layer):
        lp = list(layer.parameters())
        if lp: pgs.append({"params": lp, "lr": lr_bert*(decay**(nl-i)), "weight_decay": weight_decay, "name": f"bert_layer_{i}"})

    pp = list(model.bert.pooler.parameters()) if hasattr(model.bert, "pooler") else []
    if pp: pgs.append({"params": pp, "lr": lr_bert, "weight_decay": weight_decay, "name": "bert_pooler"})

    hp = (list(model.signal_emb.parameters()) + list(model.signal_cross_attn.parameters())
        + list(model.signal_norm.parameters()) + list(model.chunk_mha.parameters())
        + list(model.mha_norm.parameters()) + list(model.bilstm.parameters())
        + list(model.lstm_out_dropout.parameters()) + list(model.attn_layer.parameters())
        + list(model.classifier.parameters()))
    pgs.append({"params": hp, "lr": lr_head, "weight_decay": head_wd, "name": "head"})

    pgs = [g for g in pgs if g["params"]]
    log.info(f"  LLRD param groups: {len(pgs)}")
    for g in pgs:
        nt_ = sum(p.numel() for p in g["params"])
        ntr = sum(p.numel() for p in g["params"] if p.requires_grad)
        log.info(f"    {g['name']:20s}  lr={g['lr']:.2e}  wd={g['weight_decay']:.3f}  total={nt_:,}  trainable={ntr:,}")
    return AdamW(pgs)


# ══════════════════════════════════════════════════════════════════════════════
# CLASS WEIGHTS
# ══════════════════════════════════════════════════════════════════════════════
def compute_class_weights(labels_list, device):
    cnt = Counter(labels_list); n = len(labels_list); nc = len(cnt)
    w = torch.tensor([n/(nc*cnt.get(i,1)) for i in range(nc)],
                     dtype=torch.float, device=device).clamp(0.5, 2.0)
    log.info(f"  Class weights : {w.cpu().tolist()}")
    return w


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN ONE EPOCH
# ══════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, scheduler, scaler,
                accum_steps, class_weights, epoch):
    model.train()
    total_loss = n_correct = n_total = 0
    optimizer.zero_grad()
    use_rw = (class_weights is not None) and (epoch >= DEFERRED_RW_EPOCH)

    for step, batch in enumerate(tqdm(loader, desc=f"  Ep{epoch:02d}", leave=False, dynamic_ncols=True)):
        ids  = batch["chunk_input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["chunk_attention_mask"].to(DEVICE, non_blocking=True)
        cmsk = batch["chunk_mask"].to(DEVICE, non_blocking=True)
        labs = batch["label"].to(DEVICE, non_blocking=True)
        sigs = batch["signal_id"].to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            out = model(ids, mask, cmsk, sigs, labs, chunk_drop_prob=CHUNK_DROP_PROB)
            loss_raw = (F.cross_entropy(out.logits, labs, weight=class_weights,
                                        label_smoothing=LABEL_SMOOTHING, reduction="mean")
                        if use_rw else out.loss)
            loss = loss_raw / accum_steps

        scaler.scale(loss).backward()
        total_loss += loss_raw.item()
        n_correct  += (torch.argmax(out.logits, 1) == labs).sum().item()
        n_total    += len(labs)

        if (step+1) % accum_steps == 0 or (step+1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()

    return total_loss / len(loader), n_correct / n_total


# ══════════════════════════════════════════════════════════════════════════════
# EVALUATE
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(model, loader, class_weights=None, epoch=0):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    use_rw = (class_weights is not None) and (epoch >= DEFERRED_RW_EPOCH)

    with torch.no_grad():
        for batch in tqdm(loader, desc="  eval", leave=False, dynamic_ncols=True):
            ids  = batch["chunk_input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["chunk_attention_mask"].to(DEVICE, non_blocking=True)
            cmsk = batch["chunk_mask"].to(DEVICE, non_blocking=True)
            labs = batch["label"].to(DEVICE, non_blocking=True)
            sigs = batch["signal_id"].to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):
                out = model(ids, mask, cmsk, sigs, labs, chunk_drop_prob=0.0)
                loss_raw = (F.cross_entropy(out.logits, labs, weight=class_weights,
                                            label_smoothing=LABEL_SMOOTHING)
                            if use_rw else out.loss)

            total_loss += loss_raw.item()
            probs = torch.softmax(out.logits.float(), 1).cpu().tolist()
            preds = torch.argmax(out.logits, 1).cpu().tolist()
            all_preds.extend(preds); all_labels.extend(labs.cpu().tolist())
            all_probs.extend([p[1] for p in probs])

    f1_cls = f1_score(all_labels, all_preds, average=None, zero_division=0)
    dist   = Counter(all_preds)
    if len(dist) < 2: log.warning(f"  ⚠️  Class collapse: {dict(dist)}")

    try:    auc = roc_auc_score(all_labels, all_probs)
    except: auc = 0.0
    try:    mcc = matthews_corrcoef(all_labels, all_preds)
    except: mcc = 0.0
    try:    kap = cohen_kappa_score(all_labels, all_preds)
    except: kap = 0.0

    return {
        "loss"    : total_loss / len(loader),
        "acc"     : accuracy_score(all_labels, all_preds),
        "f1"      : f1_score(all_labels, all_preds, average="macro",  zero_division=0),
        "prec"    : precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "rec"     : recall_score(all_labels, all_preds, average="macro",    zero_division=0),
        "auc"     : auc, "mcc": mcc, "kappa": kap,
        "f1_rej"  : float(f1_cls[0]) if len(f1_cls) > 0 else 0.0,
        "f1_acc"  : float(f1_cls[1]) if len(f1_cls) > 1 else 0.0,
        "preds"   : all_preds, "labels": all_labels, "probs": all_probs,
        "pred_dist": dict(dist),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ADAPTIVE HP  (v2: no unfreeze)
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveHPController:
    def __init__(self):
        self.overfit_streak = self.dropout_bumped = self.wd_bumped = 0

    def step(self, epoch, train_loss, val_loss, val_f1, model, optimizer):
        actions = []
        self.overfit_streak = self.overfit_streak+1 if val_loss-train_loss > OVERFIT_GAP_THRESH else 0

        if self.overfit_streak >= OVERFIT_PATIENCE:
            if not self.dropout_bumped:
                for m in model.modules():
                    if isinstance(m, nn.Dropout): m.p = min(m.p+0.05, 0.55)
                self.dropout_bumped = True
                dp = [m.p for m in model.modules() if isinstance(m, nn.Dropout)]
                actions.append(f"dropout→{dp[0]:.2f}")
            elif not self.wd_bumped:
                for pg in optimizer.param_groups: pg["weight_decay"] = min(pg["weight_decay"]*1.5, 0.15)
                self.wd_bumped = True; actions.append("weight_decay bumped")

        if epoch >= 8 and val_f1 < UNDERFIT_F1_THRESH:
            log.info(f"  [AdaptiveHP ep{epoch}] Underfitting detected — auto-unfreeze DISABLED")

        if actions: log.info(f"  [AdaptiveHP ep{epoch}] {' | '.join(actions)}")
        return actions


# ══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ══════════════════════════════════════════════════════════════════════════════
def save_plots(history, labels, preds, swa_start):
    ep = [h["epoch"] for h in history]
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle("InCaseLawBERT v2-fast — Anti-Overfit", fontsize=13, fontweight="bold")

    ax = axes[0,0]
    ax.plot(ep, [h["train_loss"] for h in history], "b-o", ms=4, label="Train Loss")
    ax.plot(ep, [h["val_loss"]   for h in history], "r-o", ms=4, label="Val Loss")
    if swa_start <= max(ep): ax.axvline(swa_start, color="orange", ls="--", alpha=0.7, label=f"SWA@{swa_start}")
    ax.fill_between(ep, [abs(h["val_loss"]-h["train_loss"]) for h in history], alpha=0.15, color="red")
    ax.set_title("Loss Curve"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0,1]
    ax.plot(ep, [h["val_f1"]   for h in history], "g-s", ms=4, label="Val Macro-F1")
    ax.plot(ep, [h["train_acc"] for h in history], "b-s", ms=4, label="Train Acc")
    ax.plot(ep, [h["val_acc"]  for h in history], "r-s", ms=4, label="Val Acc")
    ax.axhline(SOTA_F1, color="purple", ls="--", label=f"SOTA={SOTA_F1}")
    ax.set_title("F1 / Accuracy"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0,1)

    ax = axes[0,2]
    ax.plot(ep, [h["val_auc"] for h in history], "m-^", ms=4, label="AUC")
    ax.plot(ep, [h["val_mcc"] for h in history], "c-^", ms=4, label="MCC")
    ax.axhline(0.5, color="gray", ls=":", alpha=0.5)
    ax.set_title("AUC & MCC"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1,0]
    ax.plot(ep, [h["val_f1_rej"] for h in history], "r-o", ms=4, label="F1 REJECTED")
    ax.plot(ep, [h["val_f1_acc"] for h in history], "g-o", ms=4, label="F1 ACCEPTED")
    ax.set_title("Per-class F1"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0,1)

    ax = axes[1,1]
    gap = [h["val_loss"]-h["train_loss"] for h in history]
    ax.plot(ep, gap, "k-o", ms=4)
    ax.axhline(OVERFIT_GAP_THRESH, color="red", ls="--", label=f"thresh={OVERFIT_GAP_THRESH}")
    ax.axhline(0, color="gray", ls=":")
    ax.fill_between(ep, gap, 0, where=[g>0 for g in gap], alpha=0.2, color="red",  label="Overfit")
    ax.fill_between(ep, gap, 0, where=[g<=0 for g in gap], alpha=0.2, color="blue", label="Underfit")
    ax.set_title("Train-Val Loss Gap"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1,2]
    cm = confusion_matrix(labels, preds)
    im = ax.imshow(cm, cmap="Blues"); plt.colorbar(im, ax=ax)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["REJECTED","ACCEPTED"]); ax.set_yticklabels(["REJECTED","ACCEPTED"])
    ax.set_title("Confusion Matrix — Best Epoch")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i][j]), ha="center", va="center", fontsize=12, fontweight="bold",
                    color="white" if cm[i][j]>cm.max()/2 else "black")

    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/full_analysis_v2.png", dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"  Plots → {PLOT_DIR}/full_analysis_v2.png")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN  — no `if __name__ == "__main__":` guard (Jupyter safe)
# ══════════════════════════════════════════════════════════════════════════════

log.info("="*60 + "\n  LOADING DATA\n" + "="*60)
records = load_data(INPUT_PATH)
log.info(f"  QA pairs : {len(records):,}  |  Docs : {len(set(r['doc_id'] for r in records)):,}")
log.info(f"  Source dist : {dict(Counter(r.get('source','?') for r in records))}")

pool_records, pool_ids = build_balanced_pool(records, max_docs=MAX_TOTAL_DOCS, seed=SEED)
train_records, val_records, n_tr, n_va = split_train_val(pool_records, pool_ids, seed=SEED)

# ── Tokeniser ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(INLEGAL_MODEL_ID)
if WITH_SIGNAL:
    tokenizer.add_tokens(["[FP]", "[FR]", "[N]"])
    log.info(f"  Vocab size : {len(tokenizer):,}")

# ── Datasets ──────────────────────────────────────────────────────────────────
log.info("="*60 + "\n  PRE-TOKENISING\n" + "="*60)
train_ds = HierarchicalLegalQADataset(train_records, tokenizer, MAX_CHUNK_LEN, MAX_CHUNKS, WITH_SIGNAL)
val_ds   = HierarchicalLegalQADataset(val_records,   tokenizer, MAX_CHUNK_LEN, MAX_CHUNKS, WITH_SIGNAL)

doc_labels_train = [s["label"].item() for s in train_ds.samples]
cnt  = Counter(doc_labels_train)
n0, n1 = cnt.get(0,1), cnt.get(1,1)
log.info(f"  Train class dist → REJECTED={n0}  ACCEPTED={n1}")

w = torch.tensor([len(doc_labels_train)/(2.0*n0) if l==0
                  else len(doc_labels_train)/(2.0*n1)
                  for l in doc_labels_train], dtype=torch.float)
sampler       = WeightedRandomSampler(w, len(w), replacement=True)
class_weights = compute_class_weights(doc_labels_train, DEVICE)

# SPEED FIX: num_workers=0, no persistent_workers/prefetch_factor
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          collate_fn=collate_fn, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                          collate_fn=collate_fn, num_workers=0, pin_memory=False)

# ── Model ─────────────────────────────────────────────────────────────────────
log.info("="*60 + "\n  BUILDING MODEL\n" + "="*60)
model = HierarchicalInLegalBERT(
    model_id=INLEGAL_MODEL_ID, num_labels=2,
    dropout=DROPOUT, lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS,
    lstm_dropout=LSTM_DROPOUT, mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
    label_smoothing=LABEL_SMOOTHING, freeze_bert_layers=FREEZE_BERT_LAYERS,
)
if WITH_SIGNAL: model.resize_token_embeddings(len(tokenizer))
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
log.info(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

optimizer = build_llrd_optimizer(model, LR_BERT, LR_HEAD, LLRD_DECAY, WEIGHT_DECAY, HEAD_WEIGHT_DECAY)

steps_per_epoch = (len(train_loader) + ACCUM_STEPS - 1) // ACCUM_STEPS
total_steps     = steps_per_epoch * MAX_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)
log.info(f"  Steps/ep={steps_per_epoch}  total={total_steps}  warmup={warmup_steps}")

scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler(enabled=USE_AMP)

swa_model     = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5, anneal_strategy="cos")
swa_active    = False
ahp           = AdaptiveHPController()

# ── Resume ────────────────────────────────────────────────────────────────────
start_epoch = 1; best_f1 = 0.0; best_epoch = 0
best_metrics = {}; no_improve = 0; history = []

if os.path.exists(CKPT_ROLL):
    try:
        ck = torch.load(CKPT_ROLL, map_location=DEVICE)
        model.load_state_dict(ck["model_state"])
        optimizer.load_state_dict(ck["optimizer_state"])
        scheduler.load_state_dict(ck["scheduler_state"])
        scaler.load_state_dict(ck["scaler_state"])
        start_epoch = ck["epoch"]+1; best_f1 = ck["best_f1"]
        best_epoch  = ck["best_epoch"]; best_metrics = ck["best_metrics"]
        no_improve  = ck["no_improve"]; history = ck["history"]
        if ck.get("swa_state"): swa_model.load_state_dict(ck["swa_state"])
        log.info(f"  ▶ RESUMED from epoch {ck['epoch']} (best F1={best_f1:.4f})")
    except Exception as e:
        log.warning(f"  ⚠️  Checkpoint load failed: {e} — starting fresh")

# ── CSV ───────────────────────────────────────────────────────────────────────
csv_exists = os.path.exists(CSV_PATH) and start_epoch > 1
csv_file   = open(CSV_PATH, "a" if csv_exists else "w", newline="")
csv_writer = csv.writer(csv_file)
if not csv_exists:
    csv_writer.writerow(["epoch","train_loss","train_acc","val_loss","val_acc",
                         "val_f1","val_prec","val_rec","val_auc","val_mcc","val_kappa",
                         "val_f1_rej","val_f1_acc","overfit_gap","swa_active",
                         "epoch_secs","adaptive_actions"])

# ── Training loop ─────────────────────────────────────────────────────────────
log.info("="*60)
log.info(f"  TRAINING — {MAX_EPOCHS} epochs | {n_tr} train docs | {n_va} val docs")
log.info(f"  Expected ~{steps_per_epoch} steps/ep  (was ~237 in v2 slow, now ~{steps_per_epoch})")
log.info("="*60)

start_time = datetime.now()

for epoch in range(start_epoch, MAX_EPOCHS+1):
    ep_start = datetime.now()

    if epoch >= SWA_START and not swa_active:
        swa_active = True
        log.info(f"  SWA activated at epoch {epoch}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler,
                                        scaler, ACCUM_STEPS, class_weights, epoch)
    val_m = evaluate(model, val_loader, class_weights, epoch)

    if swa_active:
        swa_model.update_parameters(model)
        swa_scheduler.step()

    actions  = ahp.step(epoch, train_loss, val_m["loss"], val_m["f1"], model, optimizer)
    ep_secs  = (datetime.now() - ep_start).total_seconds()
    done_min = (datetime.now() - start_time).total_seconds() / 60
    eta_min  = ep_secs * (MAX_EPOCHS - epoch) / 60
    gap      = val_m["loss"] - train_loss

    log.info(
        f"  Ep {epoch:02d}/{MAX_EPOCHS} | "
        f"TrLoss={train_loss:.4f} TrAcc={train_acc:.4f} | "
        f"VaLoss={val_m['loss']:.4f} VaAcc={val_m['acc']:.4f} VaF1={val_m['f1']:.4f} | "
        f"AUC={val_m['auc']:.4f} MCC={val_m['mcc']:.4f} κ={val_m['kappa']:.4f} | "
        f"F1[REJ={val_m['f1_rej']:.3f} ACC={val_m['f1_acc']:.3f}] | "
        f"Gap={gap:+.4f} SWA={'✓' if swa_active else '✗'} | "
        f"{ep_secs:.0f}s elapsed={done_min:.0f}m ETA≈{eta_min:.0f}m"
    )

    history.append({"epoch": epoch,
                    "train_loss": round(train_loss,4), "train_acc": round(train_acc,4),
                    "val_loss": round(val_m["loss"],4), "val_f1": round(val_m["f1"],4),
                    "val_acc": round(val_m["acc"],4),   "val_auc": round(val_m["auc"],4),
                    "val_mcc": round(val_m["mcc"],4),   "val_f1_rej": round(val_m["f1_rej"],4),
                    "val_f1_acc": round(val_m["f1_acc"],4)})

    csv_writer.writerow([epoch, round(train_loss,4), round(train_acc,4),
                         round(val_m["loss"],4), round(val_m["acc"],4), round(val_m["f1"],4),
                         round(val_m["prec"],4), round(val_m["rec"],4), round(val_m["auc"],4),
                         round(val_m["mcc"],4),  round(val_m["kappa"],4), round(val_m["f1_rej"],4),
                         round(val_m["f1_acc"],4), round(gap,4), int(swa_active),
                         round(ep_secs,1), "|".join(actions)])
    csv_file.flush()

    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]; best_epoch = epoch; best_metrics = val_m; no_improve = 0
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "best_f1": best_f1, "val_acc": val_m["acc"],
                    "val_auc": val_m["auc"], "val_mcc": val_m["mcc"]}, CKPT_BEST)
        log.info(f"  ✅  New best F1={best_f1:.4f} → {CKPT_BEST}")
    else:
        no_improve += 1
        log.info(f"  No improve {no_improve}/{EARLY_STOP_PAT} (best F1={best_f1:.4f} @ ep {best_epoch})")

    torch.save({"epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(), "swa_state": swa_model.state_dict() if swa_active else None,
                "best_f1": best_f1, "best_epoch": best_epoch, "best_metrics": best_metrics,
                "no_improve": no_improve, "history": history}, CKPT_ROLL)

    if no_improve >= EARLY_STOP_PAT:
        log.info(f"  ⏹  Early stopping at epoch {epoch}"); break

csv_file.close()

# ── SWA BN update ─────────────────────────────────────────────────────────────
if swa_active:
    log.info("  SWA — updating BN statistics ...")
    update_bn(train_loader, swa_model, device=DEVICE)
    swa_val = evaluate(swa_model, val_loader, class_weights, MAX_EPOCHS)
    log.info(f"  SWA → F1={swa_val['f1']:.4f}  Acc={swa_val['acc']:.4f}  AUC={swa_val['auc']:.4f}")
    if swa_val["f1"] > best_f1:
        torch.save({"model_state": swa_model.state_dict(), "source": "SWA", "f1": swa_val["f1"]},
                   f"{OUTPUT_DIR}/swa_best_model.pt")
        log.info("  ✅  SWA model is best → saved")

# ── Final report ──────────────────────────────────────────────────────────────
total_mins = (datetime.now() - start_time).total_seconds() / 60
log.info("\n" + "="*60)
log.info(f"  FINAL RESULTS  (best epoch = {best_epoch})")
log.info("="*60)
log.info(f"  Val Acc : {best_metrics['acc']:.4f}   SOTA={SOTA_ACC}")
log.info(f"  Val F1  : {best_metrics['f1']:.4f}   SOTA={SOTA_F1}")
log.info(f"  Val AUC : {best_metrics['auc']:.4f}")
log.info(f"  Val MCC : {best_metrics['mcc']:.4f}")
log.info(f"  Val κ   : {best_metrics['kappa']:.4f}")
log.info(f"  F1 REJ  : {best_metrics['f1_rej']:.4f}")
log.info(f"  F1 ACC  : {best_metrics['f1_acc']:.4f}")
log.info(f"  Runtime : {total_mins:.1f} min")
log.info(f"\n{classification_report(best_metrics['labels'], best_metrics['preds'], target_names=['REJECTED','ACCEPTED'], digits=4)}")

save_plots(history, best_metrics["labels"], best_metrics["preds"], SWA_START)
log.info(f"  Best model → {CKPT_BEST}")
log.info(f"  CSV        → {CSV_PATH}")
log.info(f"  Plots      → {PLOT_DIR}/full_analysis_v2.png")
log.info(f"  Log        → {log_file}")
log.info("  Done.")

17:46:32 | Device        : cuda  |  AMP: True
17:46:32 | SPEED FIX     : num_workers=0 | BATCH=16 | ACCUM=1 (eff batch=16)
17:46:32 | Architecture  : InCaseLawBERT → SignalCrossAttn → MHA(4h) → BiLSTM(128h,1L) → AttnPool → Linear
17:46:32 | Epochs        : 50  patience=15  SWA from ep 10
17:46:32 | LR BERT/HEAD  : 5e-06/5e-06  LLRD=0.95  WD=0.05  HeadWD=0.1
17:46:32 | Dropout       : main=0.4  lstm=0.3  mha=0.3
17:46:32 | MAX_CHUNKS    : 2  CHUNK_DROP=0.35  LABEL_SMOOTH=0.15
17:46:32 | FREEZE_BERT   : 10 layers  (auto-unfreeze DISABLED)
17:46:32 | ============================================================
  LOADING DATA
17:46:32 |   QA pairs : 45,329  |  Docs : 5,421
17:46:32 |   Source dist : {'FAC': 32588, 'ARG_P': 7120, 'ARG_R': 5621}
17:46:33 |   Balanced pool : 4,738 docs  (2369/class)  QA=39,604
17:46:33 |   Train : 3790 docs (31,722 QA)  |  Val : 948 docs (7,882 QA)
17:46:33 | HTTP Request: HEAD https://huggingface.co/law-ai/InCaseLawBERT/resolve/main/config.json "HTTP/1.1 307

  tokenising:   0%|          | 0/3790 [00:00<?, ?it/s]

17:46:39 |   Dataset ready : 3790 docs (pre-tokenised)


  tokenising:   0%|          | 0/948 [00:00<?, ?it/s]

17:46:40 |   Dataset ready : 948 docs (pre-tokenised)
17:46:40 |   Train class dist → REJECTED=2130  ACCEPTED=1660
17:46:42 |   Class weights : [0.8896713852882385, 1.141566276550293]
17:46:42 | ============================================================
  BUILDING MODEL
17:46:42 | HTTP Request: HEAD https://huggingface.co/law-ai/InCaseLawBERT/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
17:46:42 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/law-ai/InCaseLawBERT/7f2c2a0c1ff4149e8c4a8c79ee9f24757ad5dacd/config.json "HTTP/1.1 200 OK"
17:46:45 | HTTP Request: HEAD https://huggingface.co/law-ai/InCaseLawBERT/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
17:46:45 | HTTP Request: GET https://huggingface.co/api/models/law-ai/InCaseLawBERT "HTTP/1.1 200 OK"


...

  Ep20:   0%|                                                                                   | 0/237 [00:00…

  eval:   0%|                                                                                    | 0/30 [00:00…

17:52:57 |   [AdaptiveHP ep20] Underfitting detected — auto-unfreeze DISABLED
17:52:57 |   Ep 20/50 | TrLoss=0.6751 TrAcc=0.5588 | VaLoss=0.6203 VaAcc=0.6698 VaF1=0.5238 | AUC=0.5456 MCC=0.0529 κ=0.0522 | F1[REJ=0.260 ACC=0.788] | Gap=-0.0549 SWA=✓ | 17s elapsed=6m ETA≈8m
17:52:57 |   No improve 15/15 (best F1=0.5311 @ ep 5)
17:52:58 |   ⏹  Early stopping at epoch 20
17:52:58 |   SWA — updating BN statistics ...


  eval:   0%|                                                                                    | 0/30 [00:00…

17:53:00 |   SWA → F1=0.5109  Acc=0.6762  AUC=0.5405
17:53:00 | 
17:53:00 |   FINAL RESULTS  (best epoch = 5)
17:53:00 | ============================================================
17:53:00 |   Val Acc : 0.6878   SOTA=0.78
17:53:00 |   Val F1  : 0.5311   SOTA=0.8131
17:53:00 |   Val AUC : 0.5419
17:53:00 |   Val MCC : 0.0738
17:53:00 |   Val κ   : 0.0716
17:53:00 |   F1 REJ  : 0.2600
17:53:00 |   F1 ACC  : 0.8021
17:53:00 |   Runtime : 6.2 min
17:53:00 | 
              precision    recall  f1-score   support

    REJECTED     0.3230    0.2176    0.2600       239
    ACCEPTED     0.7624    0.8463    0.8021       709

    accuracy                         0.6878       948
   macro avg     0.5427    0.5319    0.5311       948
weighted avg     0.6516    0.6878    0.6655       948

17:53:02 |   Plots → single_run_results_INcaselawbert/plots/full_analysis_v2.png
17:53:02 |   Best model → single_run_results_INcaselawbert/best_model.pt
17:53:02 |   CSV        → single_run_results_INcaselawbert